#  Getting Started with AIR·MS: Health Data Fundamentals

This notebook is a companion to the AIRMS training session. It demonstrates:

- Connecting to AIRMS with the **`airms-connect`** library  
- Navigating OMOP concepts, mappings, and hierarchies  
- Extracting an atrial fibrillation (AF) cohort and deriving features:
  - Pacemaker placement  
  - Antiplatelet therapy at AF index  
  - BMI near AF diagnosis  
  - Age, sex  
- Generating de-identified summaries and figures suitable for exploratory data analyses and grant submissions

---

### Prerequisites to run this notebook
Before you can execute queries, you need:

1. A valid **Mount Sinai school network account**  
2. A **Minerva HPC account** (apply separately)  
3. **Approved access** to the AIRMS CDMDEID dataset (via SailPoint, linked to IRB #20-01288)  
4. Connection to the **Mount Sinai network** (on campus or via VPN)  

We strongly recommend launching Jupyter through **Open OnDemand** on Minerva (see the training PowerPoint for setup instructions). [OnDemand Getting Started.pptx](https://mtsinai-my.sharepoint.com/:p:/g/personal/eugeniaalessandrae_allevabonomi_mssm_edu/EbnO2s2KPHRErKG8sgUdDncBjZVH-AyyCDIVIM8GkiRndA?e=CAZdtb)



## Connecting to AIR-MS 

NB: If you are running this notebook from your local device or using your own conda environment, you will need to install the airms-connect library via
``` bash
pip install airms-connect --index-url=https://airms_python_packages_test:7olj5EEtEPOdd5IZNFCQIvjUiq5MrpOGQx5QA6MXoOfMjohdhZzkJQQJ99BEACAAAAAg3d00AAASAZDOyw9m@pkgs.dev.azure.com/airms/16abab1a-0056-4bff-a637-0fe034e604c4/_packaging/airms_python_packages_test/pypi/simple/
```

In [ ]:
# Install dotenv (if required)
%pip install python-dotenv

# AIRMS connect import
from airms_connect.connection import airms_connection

# Other imports
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
from textwrap import fill

In [ ]:
# AIRMS connection

airms = airms_connection()

# On Minerva: establish the tunnel automatically.
# (Default for training; comment out if running locally)
airms.on_minerva(login_host_name="li04e01")

# Connect
airms.connect()


In [ ]:
# Quick smoke test: peek at PERSON table
sql = """
SELECT TOP 5 person_id, gender_concept_id, year_of_birth
FROM CDMDEID.PERSON
"""
airms.conn.sql(sql).collect()

## Patient Identifiers 

In the **CDMDEID schema** (de-identified OMOP):
- Each patient is represented by a surrogate **`person_id`**.  
- `person_id` is stable within the schema and is the **primary join key** across all OMOP tables (`condition_occurrence`, `procedure_occurrence`, `drug_exposure`, `measurement`, etc.).

In the **PHI schema** (CDM):
- An extension column **`XTN_EPIC_MRN`** stores the EPIC medical record number (MRN).  
- This is only available when IRB approval explicitly allows PHI use.  
- For de-identified analytics and grant-prep work, you will typically work with `person_id` only.

**Best practice:**  
- Always use `person_id` for joining tables.  
- Use `XTN_EPIC_MRN` *only* within PHI-approved projects where direct linkage to the EPIC MRN is necessary (e.g., recruitment, chart review).  

In [ ]:
sql = """
SELECT TOP 5
    person_id,
    gender_concept_id,
    year_of_birth,
    race_concept_id,
    ethnicity_concept_id
FROM CDMDEID.PERSON
"""
airms.conn.sql(sql).collect()

You can see that the PERSON_ID column stores the internal ID for each patient that allows us to link different tables from the EHR. 
This is **NOT** the Medical Record Number (MRN) - as this number is considered personal health information and is not available in the DE-ID version. 

## OMOP Concepts and Hierarchy

The **OMOP Common Data Model (CDM)** standardizes clinical data, but source systems (like EPIC) use many vocabularies (ICD10CM, CPT4, NDC, etc.).
A very useful resources to find relevant concepts and their OMOP mapping is the [Athena Browser](https://athena.ohdsi.org/)

Key tables for working with concepts:

- **CONCEPT**  
  Stores all concepts (standard + non-standard).  
  - `standard_concept = 'S'` → Standard OMOP concept  
  - Non-standard codes (e.g., ICD, CPT, NDC) must be mapped

- **CONCEPT_RELATIONSHIP**  
  Defines pairwise relationships.  
  - Use `"Maps to"` to map non-standard → standard  
  - Example: ICD10CM `I48.0` (Paroxysmal AF) → SNOMED `49436004` (Paroxysmal AF) ([Link](https://athena.ohdsi.org/search-terms/terms/313217) to Athena page)

- **CONCEPT_ANCESTOR**  
  Encodes hierarchies (parent–child relationships).  
  - Lets you expand from a parent concept to all its descendants  
  - Example: “Atrial fibrillation” (SNOMED parent) → all subtypes (paroxysmal, persistent, chronic…)

---

### Concept set identification strategies

When defining concept sets in OMOP, there are three complementary approaches.  
We will use all three together:

1. **Approach A – Recon by Name (exploration):**  
   Search the `CONCEPT` table by keyword to get a sense of possible terms.  
   This is useful for discovery, but not reliable enough for analysis.

2. **Approach B – Map Non-standard → Standard (authoritative step):**  
   Start from source vocabularies used in EPIC (ICD10CM, CPT4, NDC, etc.)  
   and use `CONCEPT_RELATIONSHIP` with `relationship_id = 'Maps to'` to map to **standard concepts** (SNOMED, RxNorm).  
   These standard concepts are the backbone of OMOP analysis.

3. **Approach C – Expand via Hierarchy (comprehensive coverage):**  
   Use `CONCEPT_ANCESTOR` to include all descendant concepts under a chosen standard parent.  
   This ensures you capture all subtypes and coding variation.

**Training pattern we’ll follow:**  
A (explore) → B (map to standard) → C (expand descendants).  
This ensures we capture *all* relevant events and avoid missing data due to coding variation.

### Quality Control - Collaborating with Clinicians is a must!
Even though OMOP standardization helps, you should always confirm with a **clinical collaborator** that:
- All source codes included are relevant, and
- No important codes were missed.  

This review step is crucial for defensibility of your cohort definition.

In [ ]:
# Approach A
# Quick scan of atrial fibrillation by name

condition_name = '%atrial fibrillation%'

sql = f"""
SELECT TOP 20
    concept_id,
    concept_name,
    vocabulary_id,
    domain_id,
    standard_concept
FROM CDMDEID.CONCEPT
WHERE LOWER(concept_name) LIKE '{condition_name}'
ORDER BY standard_concept DESC, vocabulary_id, concept_name
"""
airms.conn.sql(sql).collect()

In [ ]:
# Approach B
# ICD10CM I48* AF codes -> SNOMED standard concepts

code = 'I48%'
vocabulary = 'ICD10CM'
domain = 'Condition'
relationship = 'Maps to'

sql_maps_to = f"""
SELECT
    c1.concept_code    AS source_code,
    c1.concept_name    AS source_name,
    c1.vocabulary_id   AS source_vocab,
    c2.concept_id      AS standard_concept_id,
    c2.concept_name    AS standard_name,
    c2.vocabulary_id   AS standard_vocab
FROM CDMDEID.CONCEPT c1
JOIN CDMDEID.CONCEPT_RELATIONSHIP cr
  ON cr.concept_id_1 = c1.concept_id
JOIN CDMDEID.CONCEPT c2
  ON c2.concept_id = cr.concept_id_2
WHERE c1.vocabulary_id = '{vocabulary}'
  AND c1.concept_code LIKE '{code}'
  AND cr.relationship_id = '{relationship}'
  AND c2.standard_concept = 'S'
  AND c2.domain_id = '{domain}'
"""
mapped_af = airms.conn.sql(sql_maps_to).collect()
mapped_af

In [ ]:
# Approach C
# Use mapped SNOMED parents -> expand to descendants
sql_af_conceptset = f"""
WITH mapped_snomed AS (
  {sql_maps_to}
)
SELECT DISTINCT
    ca.descendant_concept_id              AS concept_id,
    c.concept_name                        AS concept_name,
    c.vocabulary_id                       AS vocabulary_id,
    c.domain_id                           AS domain_id,
    c.standard_concept                    AS standard_flag
FROM CDMDEID.CONCEPT_ANCESTOR ca
JOIN CDMDEID.CONCEPT c
  ON c.concept_id = ca.descendant_concept_id
WHERE ca.ancestor_concept_id IN (
  SELECT standard_concept_id FROM mapped_snomed
)
ORDER BY c.vocabulary_id, c.concept_name
"""
af_concepts = airms.conn.sql(sql_af_conceptset).collect()
af_concepts

In [ ]:
# QUALITY CONTROL STEP
# Identify which source concepts (ICD10CM etc.) actually contributed
# to the atrial fibrillation (AF) cohort in CONDITION_OCCURRENCE.
# This list should be reviewed by a clinical collaborator to confirm
# that all included codes are relevant, and that nothing important
# was missed.

sql_sources_for_review = f"""
WITH af_cs AS (
  {sql_af_conceptset}
),
af_hits AS (
  SELECT
      co.condition_source_concept_id
  FROM CDMDEID.CONDITION_OCCURRENCE co
  WHERE co.condition_concept_id IN (SELECT concept_id FROM af_cs)
    AND co.condition_source_concept_id IS NOT NULL
    AND co.condition_source_concept_id LIKE_REGEXPR '^[0-9]+$'
)
SELECT
    c.concept_id                 AS source_concept_id,
    c.concept_code               AS source_code,
    c.concept_name               AS source_name,
    c.vocabulary_id              AS source_vocab,
    COUNT(*)                     AS n_occurrences
FROM af_hits h
JOIN CDMDEID.CONCEPT c
  ON c.concept_id = TO_INTEGER(h.condition_source_concept_id)
GROUP BY c.concept_id, c.concept_code, c.concept_name, c.vocabulary_id
ORDER BY n_occurrences DESC, source_vocab, source_code
"""

af_source_review = airms.conn.sql(sql_sources_for_review).collect()
af_source_review

# Optional: export to CSV for clinical review
# af_source_review..to_csv("af_source_codes_for_clinical_review.csv", index=False)

## Extracting the Atrial Fibrillation Cohort - Condition Occurrence Table

Once we’ve defined the AF concept set (Approach A → B → C + QC), the next step is to
identify **cohort** of patients with Atrial Fibrillation and the **index date** (First AF diagnosis) for each patient:

- Query `CONDITION_OCCURRENCE` for rows where `condition_concept_id` is in our AF concept set.  
- Take the **earliest `condition_start_date`** per patient.  
- This will serve as the cohort’s **AF index date**, anchoring all downstream analyses
  (time to pacemaker, drug exposure at index, BMI near index, stroke follow-up, etc.).

In [ ]:
# Build AF index dates: earliest AF diagnosis per person
sql_af_index = f"""
WITH af_concepts AS (
  {sql_af_conceptset}
)
SELECT
    co.person_id,
    MIN(co.condition_start_date) AS af_index_date
FROM CDMDEID.CONDITION_OCCURRENCE co
WHERE co.condition_concept_id IN (SELECT concept_id FROM af_concepts)
GROUP BY co.person_id
"""

af_index = airms.conn.sql(sql_af_index).collect()
af_index.head(10)

In [ ]:
# How many patients have AF?
print("AF cohort size:", len(af_index))

# Make a year column for visualization
af_index["year"] = pd.to_datetime(af_index["AF_INDEX_DATE"]).dt.year

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
ax = af_index["year"].value_counts().sort_index().plot(
    kind="bar",
    color="#1f77b4",
    edgecolor="black"
)
plt.title("Number of patients with first AF diagnosis per year", fontsize=14, pad=15)
plt.xlabel("Year of AF index", fontsize=12)
plt.ylabel("Patient count", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

## Pacemaker implantation procedure - Procedure Occurrence Table
In this section we identify **pacemaker implantation procedures** in OMOP.  
Finally, we’ll derive each patient’s **first pacemaker date after their AF index date**, and compute **days from AF index → pacemaker implantation**.  
This variable becomes an important feature for downstream analyses.

In [ ]:
# Approach A: text search
name = '%pacemaker%'
domain = 'Procedure'

sql_pm_recon = f"""
SELECT TOP 20
    concept_id,
    concept_name,
    vocabulary_id,
    domain_id,
    standard_concept
FROM CDMDEID.CONCEPT
WHERE domain_id = '{domain}'
  AND LOWER(concept_name) LIKE '{name}'
ORDER BY standard_concept DESC, vocabulary_id, concept_name
"""
airms.conn.sql(sql_pm_recon).collect()

In [ ]:
# Let's say we are satisfied with all codes identified above. Let's see if we have additional sub-codes to extract 
sql_pm_conceptset = f"""
WITH mapped_std AS (
  {sql_pm_recon}
)
SELECT DISTINCT
    ca.descendant_concept_id   AS concept_id,
    c.concept_name             AS concept_name,
    c.vocabulary_id            AS vocabulary_id,
    c.domain_id                AS domain_id,
    c.standard_concept         AS standard_flag
FROM CDMDEID.CONCEPT_ANCESTOR ca
JOIN CDMDEID.CONCEPT c
  ON c.concept_id = ca.descendant_concept_id
WHERE ca.ancestor_concept_id IN (SELECT concept_id FROM mapped_std)
ORDER BY c.vocabulary_id, c.concept_name
"""
pm_concepts = airms.conn.sql(sql_pm_conceptset).collect()
pm_concepts

In [ ]:
# Extract pacemaker placement occurrence and date
# This step REUSES the AF index query you defined earlier as `sql_af_index`.
# It should be a SELECT returning: person_id, af_index_date

sql_pm_after_af = f"""
WITH af_index AS (
  {sql_af_index}
),
pm_concepts AS (
  {sql_pm_conceptset}
),
pm_events AS (
  SELECT po.person_id, po.procedure_date
  FROM CDMDEID.PROCEDURE_OCCURRENCE po
  WHERE po.procedure_concept_id IN (SELECT concept_id FROM pm_concepts)
),
first_pm_after AS (
  SELECT a.person_id, MIN(p.procedure_date) AS first_pm_date
  FROM af_index a
  JOIN pm_events p ON p.person_id = a.person_id
   AND p.procedure_date >= a.af_index_date
  GROUP BY a.person_id
)
SELECT
    a.person_id,
    a.af_index_date,
    f.first_pm_date,
    CASE WHEN f.first_pm_date IS NOT NULL
         THEN DAYS_BETWEEN(a.af_index_date, f.first_pm_date)
    END AS days_to_pacemaker
FROM af_index a
LEFT JOIN first_pm_after f ON f.person_id = a.person_id
"""
pm_after_af = airms.conn.sql(sql_pm_after_af).collect()
pm_after_af.head()

In [ ]:
# Plot pacemaker implantation
df = pm_after_af.copy()

# Ensure expected column names (HANA usually uppercases)
cols = {c.upper(): c for c in df.columns}
# Standardize references
pcol  = 'PERSON_ID' if 'PERSON_ID' in cols else 'person_id'
icol  = 'AF_INDEX_DATE' if 'AF_INDEX_DATE' in cols else 'af_index_date'
pmcol = 'FIRST_PM_DATE' if 'FIRST_PM_DATE' in cols else 'first_pm_date'
dcol  = 'DAYS_TO_PACEMAKER' if 'DAYS_TO_PACEMAKER' in cols else 'days_to_pacemaker'


# Histogram: days to pacemaker (only those with pacemaker)
days = df.loc[df[pmcol].notna(), dcol].dropna()

plt.figure(figsize=(9, 5))
plt.hist(days, bins=40)
plt.title('Distribution: days from AF index to first pacemaker', fontsize=13, pad=12)
plt.xlabel('Days to pacemaker', fontsize=11)
plt.ylabel('Patient count', fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.6)
# Optional: cap x-axis to focus on typical range (uncomment to adjust)
# plt.xlim(0, 365*3)  # e.g., focus on first 3 years
plt.tight_layout()
plt.show()

# quick text summary
n_with = sum(df.FIRST_PM_DATE.isna())
n_without = sum(~df.FIRST_PM_DATE.isna())
print(f"Patients with pacemaker: {n_with:,}")
print(f"Patients without pacemaker: {n_without:,}")
if len(days) > 0:
    print(
        f"Days to pacemaker — n={len(days):,}, "
        f"median={int(days.median())}, "
        f"IQR=({int(days.quantile(0.25))}, {int(days.quantile(0.75))}), "
        f"min={int(days.min())}, max={int(days.max())}"
    )

## Antiplatelet Therapy at AF Index - Drug Exposure Table

Instead of listing individual drug codes, we leverage **therapeutic class hierarchy**:

- Use the **ATC** class **“Antiplatelet agent”** as the ancestor (concept_id = [35807468](https://athena.ohdsi.org/search-terms/terms/35807468)).
- From that ancestor, extract **RxNorm Ingredients** (children) via `CONCEPT_ANCESTOR`.
- Then expand each **Ingredient → all descendant RxNorm drug concepts** (Clinical Drug, Branded Drug, etc.) for use in `DRUG_EXPOSURE`.

This approach is maintainable and comprehensive:
1) **ATC ancestor → Ingredients** (what molecules belong to the class?)  
2) **Ingredients → RxNorm descendants** (all formulations/brands used in exposure data)  
3) Join to `DRUG_EXPOSURE` and time-window relative to the AF index date.

We’ll flag patients **with any antiplatelet exposure within 6 months after AF index**.

In [ ]:
# Confirm the ancestor concept (ATC class)

antiplatelet_concept = 35807468

sql_ap_atc_info = f"""
SELECT concept_id, concept_name, vocabulary_id, concept_class_id, domain_id
FROM CDMDEID.CONCEPT
WHERE concept_id = {antiplatelet_concept}
"""
airms.conn.sql(sql_ap_atc_info).collect()

In [ ]:
# Pull RxNorm Ingredients that descend from the ATC Antiplatelet class (concept_id = 35101523)
vocabulary = 'RxNorm'
concept_class = 'Ingredient'

sql_ap_ingredients_from_atc = f"""
SELECT
    ca.max_levels_of_separation,
    c.concept_id          AS ingredient_id,
    c.concept_name        AS ingredient_name,
    c.vocabulary_id,
    c.concept_class_id,
    c.domain_id,
    c.standard_concept
FROM CDMDEID.CONCEPT_ANCESTOR ca
JOIN CDMDEID.CONCEPT c
  ON c.concept_id = ca.descendant_concept_id
WHERE ca.ancestor_concept_id = {antiplatelet_concept}   
   AND c.vocabulary_id = '{vocabulary}'
   AND c.concept_class_id = '{concept_class}'
   AND c.invalid_reason IS NULL
ORDER BY ca.max_levels_of_separation, c.concept_name
"""
ap_ingredients = airms.conn.sql(sql_ap_ingredients_from_atc).collect()
ap_ingredients

In [ ]:
# From the RxNorm Ingredients, expand to all descendant RxNorm *drug concepts* you would see in DRUG_EXPOSURE.
# We exclude the Ingredient class and keep standard RxNorm drug classes (e.g., Clinical Drug, Branded Drug).

domain = 'Drug'

sql_ap_conceptset = f"""
WITH ap_ing AS (
  {sql_ap_ingredients_from_atc}
)
SELECT DISTINCT
    ca.descendant_concept_id   AS concept_id,
    c.concept_name             AS concept_name,
    c.vocabulary_id            AS vocabulary_id,
    c.domain_id                AS domain_id,
    c.concept_class_id         AS concept_class_id,
    c.standard_concept         AS standard_flag
FROM CDMDEID.CONCEPT_ANCESTOR ca
JOIN CDMDEID.CONCEPT c
  ON c.concept_id = ca.descendant_concept_id
WHERE ca.ancestor_concept_id IN (SELECT ingredient_id FROM ap_ing)
  AND c.vocabulary_id = '{vocabulary}'
  AND c.standard_concept = 'S'
  AND c.domain_id = '{domain}'
  AND c.concept_class_id <> '{concept_class}'
ORDER BY c.concept_class_id, c.concept_name
"""
ap_concepts = airms.conn.sql(sql_ap_conceptset).collect()
ap_concepts

In [ ]:
# Flags exposure window overlap with [AF_INDEX_DATE, AF_INDEX_DATE + 6 months]
# End date computed from DRUG_EXPOSURE_END_DATE or START_DATE + DAYS_SUPPLY when end is null.

sql_pm_af_ap = f"""
WITH cohort_base AS (
  {sql_pm_after_af}
),
ap_concepts AS (
  {sql_ap_conceptset}
),
dex AS (
  SELECT d.person_id,
         d.drug_concept_id,
         d.drug_exposure_start_date,
         COALESCE(
           d.drug_exposure_end_date,
           ADD_DAYS(d.drug_exposure_start_date, COALESCE(d.days_supply,0))
         ) AS drug_exposure_end_date
  FROM CDMDEID.DRUG_EXPOSURE d
  WHERE d.drug_concept_id IN (SELECT concept_id FROM ap_concepts)
),
ap_in_6mo AS (
  SELECT
      c.person_id,
      MIN(dex.drug_exposure_start_date) AS first_ap_start_in_window,
      MAX(dex.drug_exposure_end_date)   AS last_ap_end_in_window
  FROM cohort_base c
  JOIN dex ON dex.person_id = c.person_id
   AND dex.drug_exposure_start_date <= ADD_MONTHS(c.af_index_date, 6)
   AND dex.drug_exposure_end_date   >= c.af_index_date
  GROUP BY c.person_id
)
SELECT
    c.*,
    CASE WHEN ap.first_ap_start_in_window IS NOT NULL THEN 1 ELSE 0 END AS on_antiplatelet_within_6mo,
    ap.first_ap_start_in_window,
    ap.last_ap_end_in_window
FROM cohort_base c
LEFT JOIN ap_in_6mo ap ON ap.person_id = c.person_id
"""
pm_af_ap = airms.conn.sql(sql_pm_af_ap).collect()
pm_af_ap.head()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pm_af_ap.copy()

# Standardize column names (HANA uses uppercases)
cols = {c.lower(): c for c in df.columns}
pcol  = cols.get('person_id', 'person_id')
icol  = cols.get('af_index_date', 'af_index_date')
pmcol = cols.get('first_pm_date', 'first_pm_date')
dcol  = cols.get('days_to_pacemaker', 'days_to_pacemaker')
apcol = cols.get('on_antiplatelet_within_6mo', 'on_antiplatelet_within_6mo')


# Antiplatelet within 6 months
ap_flag = (df[apcol] == 1)
ap_counts = ap_flag.value_counts().reindex([True, False], fill_value=0)
ap_counts.index = ['Antiplatelet within 6 months', 'No antiplatelet in 6 months']

plt.figure(figsize=(7, 5))
ax = ap_counts.plot(kind='bar', edgecolor='black')
plt.title('AF patients: antiplatelet therapy within 6 months')
plt.ylabel('Number of patients')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Histogram: days to pacemaker, stratified by antiplatelet status
has_pm = df[pmcol].notna()
if has_pm.any():
    fig, ax = plt.subplots(figsize=(9, 5))
    days_ap  = df.loc[has_pm & ap_flag, dcol].dropna()
    days_no  = df.loc[has_pm & (~ap_flag), dcol].dropna()

    # Plot two distributions; bins shared for comparability
    bins = np.histogram_bin_edges(pd.concat([days_ap, days_no]), bins=40)
    ax.hist(days_ap, bins=bins, alpha=0.6, label='AP within 6 mo')
    ax.hist(days_no, bins=bins, alpha=0.6, label='No AP within 6 mo')

    ax.set_title('Days from AF index to first pacemaker (by AP ≤6 mo)')
    ax.set_xlabel('Days to pacemaker')
    ax.set_ylabel('Patient count')
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    ax.legend()
    # Optional: focus x-axis range
    # ax.set_xlim(0, 365*3)
    plt.tight_layout()
    plt.show()

    # Quick summary stats
    def summary(x):
        return {
            'n': int(len(x)),
            'median': int(np.median(x)) if len(x) else None,
            'iqr_low': int(np.quantile(x, 0.25)) if len(x) else None,
            'iqr_high': int(np.quantile(x, 0.75)) if len(x) else None,
            'min': int(np.min(x)) if len(x) else None,
            'max': int(np.max(x)) if len(x) else None,
        }
    print("AP ≤6 mo:", summary(days_ap))
    print("No AP ≤6 mo:", summary(days_no))
else:
    print("No pacemaker events found; skipping days-to-pacemaker histogram.")

## BMI Measurement near AF Index

In OMOP, anthropometric values like BMI live in the **MEASUREMENT** table.

- Each row represents a quantitative measurement (`value_as_number`) recorded at a date (`measurement_date`).  
- Concepts come from vocabularies like **LOINC** or **SNOMED**.  

For BMI, we use the standard concept **"Body mass index"** and its descendants.

**Approach:**
1. Identify BMI concepts (LOINC + SNOMED descendants).  
2. Pull all BMI rows for AF patients.  
3. For each patient, find the **BMI value closest in time to AF index**. 

In [ ]:
# Find BMI concepts in OMOP (LOINC + SNOMED)

concept_name = 'body mass index'
concept_class = 'Clinical Observation'
domain = 'Measurement'

sql_bmi_conceptset = f"""
SELECT concept_id, concept_name, vocabulary_id, domain_id, concept_class_id
FROM CDMDEID.CONCEPT
WHERE domain_id = '{domain}'
  AND standard_concept = 'S'
  AND concept_class_id = '{concept_class}'
  AND LOWER(concept_name) LIKE '%{concept_name}%'
  AND invalid_reason IS NULL
"""
bmi_concepts = airms.conn.sql(sql_bmi_conceptset).collect()
bmi_concepts

In [ ]:
# We want the actual ration that was observed (i.e. 3038553)
sql_bmi_conceptset = 'SELECT * FROM CDMDEID.CONCEPT WHERE CONCEPT_ID=3038553'

In [ ]:
# Build on your existing cohort (sql_pm_after_af) and append BMI nearest to AF index
# Assumes sql_pm_after_af returns: person_id, af_index_date, first_pm_date, days_to_pacemaker,
#                                 on_antiplatelet_within_6mo, first_ap_start_in_window, last_ap_end_in_window

sql_pm_af_ap_bmi = f"""
WITH cohort_base AS (
  {sql_pm_af_ap}
),

bmi_cs AS (
  {sql_bmi_conceptset}
),

-- All BMI rows for patients in the cohort
bmi_meas AS (
  SELECT
      m.person_id,
      m.measurement_date,
      CASE 
        WHEN CAST(m.value_as_number AS DOUBLE) > 100 
          THEN NULL 
        ELSE CAST(m.value_as_number AS DOUBLE) 
      END AS value_as_number
  FROM CDMDEID.MEASUREMENT m
  JOIN cohort_base cb
    ON cb.person_id = m.person_id
  WHERE m.measurement_concept_id IN (SELECT concept_id FROM bmi_cs)
    AND m.value_as_number IS NOT NULL
),

-- Compute absolute days from AF index and pick the closest BMI per person
bmi_joined AS (
  SELECT
      cb.person_id,
      cb.af_index_date,
      b.measurement_date,
      b.value_as_number,
      ABS(DAYS_BETWEEN(cb.af_index_date, b.measurement_date)) AS abs_days_diff
  FROM cohort_base cb
  JOIN bmi_meas b
    ON b.person_id = cb.person_id
),
bmi_ranked AS (
  SELECT
      person_id,
      af_index_date,
      measurement_date,
      value_as_number,
      ROW_NUMBER() OVER (
        PARTITION BY person_id
        ORDER BY abs_days_diff ASC, measurement_date ASC
      ) AS rn
  FROM bmi_joined
),
bmi_near_index AS (
  SELECT
      person_id,
      value_as_number AS bmi_value,
      measurement_date AS bmi_date
  FROM bmi_ranked
  WHERE rn = 1
)

SELECT
    cb.person_id,
    cb.af_index_date,
    cb.first_pm_date,
    cb.days_to_pacemaker,
    cb.on_antiplatelet_within_6mo,
    cb.first_ap_start_in_window,
    cb.last_ap_end_in_window,
    b.bmi_value,
    b.bmi_date
FROM cohort_base cb
LEFT JOIN bmi_near_index b
  ON b.person_id = cb.person_id
ORDER BY cb.af_index_date
"""

pm_af_ap_bmi = airms.conn.sql(sql_pm_af_ap_bmi).collect()
pm_af_ap_bmi.head()

In [ ]:
import matplotlib.pyplot as plt

pm_af_ap_bmi.BMI_VALUE = pm_af_ap_bmi.BMI_VALUE.astype(float)

plt.figure(figsize=(8, 5))
pm_af_ap_bmi["BMI_VALUE"].hist(bins=40, edgecolor="black", color="#1f77b4")
plt.title("Distribution of BMI closest to AF index", fontsize=14, pad=12)
plt.xlabel("BMI (kg/m²)")
plt.ylabel("Number of patients")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

print("BMI cohort size:", len(pm_af_ap_bmi))
print("Median BMI:", round(pm_af_ap_bmi["BMI_VALUE"].median(), 1))

## Age and Sex from PERSON

We now extend the cohort with **demographics**:

- **Sex** is stored in `PERSON.gender_concept_id`.  
  We join to the CONCEPT table for a readable label.  
- **Age at AF index** is calculated from `PERSON.birth_date` relative to `AF_INDEX_DATE`.

This information is critical for describing the cohort and adjusting analyses.

In [ ]:
sql_pm_af_ap_age_sex = f"""
WITH cohort_base AS (
  {sql_pm_af_ap_bmi}
)
SELECT
    cb.*,
    p.birth_datetime,
    g.concept_name AS sex,
    FLOOR(MONTHS_BETWEEN(p.birth_datetime, cb.af_index_date) / 12) AS age_at_af
FROM cohort_base cb
JOIN CDMDEID.PERSON p
  ON p.person_id = cb.person_id
JOIN CDMDEID.CONCEPT g
  ON g.concept_id = p.gender_concept_id
ORDER BY cb.af_index_date
"""

pm_af_ap_age_sex = airms.conn.sql(sql_pm_af_ap_age_sex).collect()
pm_af_ap_age_sex.head()

## Exploratory Data Analyses

We summarize cohort size, demographics, therapies near diagnosis, device procedures, and BMI capture.  
Figures are de-identified and derived from the **de-identified OMOP** dataset.

**Cohort anchor:** first AF diagnosis (`AF_INDEX_DATE`).  
**Therapy window:** antiplatelet exposure overlapping `[AF_INDEX_DATE, +6 months]`.  
**Pacemaker outcome:** first pacemaker procedure on/after `AF_INDEX_DATE`.  
**BMI:** nearest measurement within ±180 days of `AF_INDEX_DATE`.

In [ ]:
# General variables
cohort  = pm_af_ap_age_sex
af = 'AF_INDEX_DATE'
pid = 'PERSON_ID'
age = 'AGE_AT_AF'
sex = 'SEX'
pm_d = 'FIRST_PM_DATE'
pm_days = 'FIRST_PM_DATE'


# Build by_month & cumulative counts (needed for the accrual plot/KPI span) ---

af_series = pd.to_datetime(cohort['AF_INDEX_DATE'], errors='coerce').dropna()
if len(af_series):
    by_month = (
        af_series.dt.to_period('M')
        .value_counts()
        .sort_index()
        .to_timestamp()
    )
    cum_by_month = by_month.cumsum()
else:
    # empty fallback so the plotting code still runs without errors
    by_month = pd.Series(dtype='int')
    cum_by_month = pd.Series(dtype='int')

# build stats needed for KPI tiles 
N = cohort['PERSON_ID'].nunique()

af_dates = pd.to_datetime(cohort['AF_INDEX_DATE'], errors='coerce').dropna()
if len(af_dates):
    by_month = (
        af_dates.dt.to_period('M')
        .value_counts()
        .sort_index()
        .to_timestamp()
    )
    cum_by_month = by_month.cumsum()
else:
    by_month = pd.Series(dtype=int)
    cum_by_month = pd.Series(dtype=int)

age_vals = cohort['AGE_AT_AF'].dropna().astype(float)
age_med  = np.nanmedian(age_vals) if len(age_vals) else np.nan
age_q1   = np.nanpercentile(age_vals,25) if len(age_vals) else np.nan
age_q3   = np.nanpercentile(age_vals,75) if len(age_vals) else np.nan

sex_counts = cohort['SEX'].fillna("Unknown").value_counts() if sex else pd.Series(dtype=int)

ap_rate = float(cohort['ON_ANTIPLATELET_WITHIN_6MO'].fillna(0).astype(int).mean()) 
has_pm = cohort['FIRST_PM_DATE'].notna() if pm_d else pd.Series(dtype=bool)
pm_rate = float(has_pm.mean()) 
pm_days_nonnull = cohort.loc[cohort['DAYS_TO_PACEMAKER'].notna(), 'DAYS_TO_PACEMAKER'].astype(float) 
pm_days_med = np.nanmedian(pm_days_nonnull) if len(pm_days_nonnull) else np.nan

bmi_series = cohort['BMI_VALUE'] 
bmi_avail  = float(bmi_series.notna().mean()) 
bmi_vals   = bmi_series.dropna().astype(float)

# BMI category shares for annotation 
def _bmi_bucket(x):
    if pd.isna(x): return np.nan
    if x < 18.5:   return 'Underweight (<18.5)'
    if x < 25:     return 'Normal (18.5–24.9)'
    if x < 30:     return 'Overweight (25–29.9)'
    return 'Obese (≥30)'

if 'bmi_vals' in locals() and len(bmi_vals):
    bmi_cat   = bmi_vals.apply(_bmi_bucket)
    bmi_share = (bmi_cat.value_counts(normalize=True)
                 .sort_index())
else:
    bmi_share = pd.Series(dtype=float)

In [ ]:
# aesthetics
bg = "#f9fafb"
fg = "#111827"
grid_c = "#9ca3af"
palette = plt.get_cmap("tab10").colors  # consistent scheme

def prettify_axes(ax):
    ax.set_facecolor(bg)
    ax.grid(True, axis='y', alpha=0.25, color=grid_c, linewidth=0.8)
    # lighten spines
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    for s in ["left", "bottom"]:
        ax.spines[s].set_color("#e5e7eb")
    ax.tick_params(colors=fg, labelsize=9)

# Figure Layout
fig = plt.figure(figsize=(14, 10), facecolor="white")
gs  = fig.add_gridspec(3, 3, height_ratios=[1.0, 1.2, 1.2], hspace=0.6, wspace=0.4)

# A) KPI tiles (top row)
ax_kpi1 = fig.add_subplot(gs[0, 0]); ax_kpi2 = fig.add_subplot(gs[0, 1]); ax_kpi3 = fig.add_subplot(gs[0, 2])
for ax in (ax_kpi1, ax_kpi2, ax_kpi3):
    ax.axis('off')
    ax.set_facecolor("white")
    # a thin accent bar
    ax.axhline(0.95, xmin=0, xmax=1, color=palette[0], linewidth=6, alpha=0.9, clip_on=False)

def draw_kpi(ax, title, value, subtitle=None, color=palette[0]):
    ax.text(0.0, 0.75, title, fontsize=11, fontweight='bold', ha='left', va='center', color=fg)
    ax.text(0.0, 0.28, value, fontsize=22, fontweight='bold', ha='left', va='center', color=color)
    if subtitle:
        ax.text(0.0, 0.05, subtitle, fontsize=9, ha='left', va='center', color=fg)

span_txt = None
if len(by_month):
    try:
        span_txt = f"AF accrual span: {by_month.index.min().date()} → {by_month.index.max().date()}"
    except Exception:
        span_txt = None

draw_kpi(ax_kpi1, "Cohort size", f"{N:,}", span_txt, color=palette[0])
draw_kpi(ax_kpi2, "Age at AF index", (f"{age_med:.1f} [IQR {age_q1:.1f}–{age_q3:.1f}]" if not np.isnan(age_med) else "—"),
         "Median [IQR]", color=palette[1])

ther_txt = f"Antiplatelet ≤6 mo: {ap_rate:.1%}" if not np.isnan(ap_rate) else "Antiplatelet ≤6 mo: —"
pm_txt   = f"Pacemaker after AF: {pm_rate:.1%}" if not np.isnan(pm_rate) else "Pacemaker after AF: —"
pm2_txt  = f"Median time: {pm_days_med:.0f} days" if not np.isnan(pm_days_med) else None
draw_kpi(ax_kpi3, "Therapy", ther_txt,
         f"{pm_txt}" + (f"\n{pm2_txt}" if pm2_txt else ""), color=palette[2])

# B) Middle row plots
ax1 = fig.add_subplot(gs[1, 0])  # Accrual (cumulative)
ax2 = fig.add_subplot(gs[1, 1])  # Age histogram
ax3 = fig.add_subplot(gs[1, 2])  # Sex bars

for ax in (ax1, ax2, ax3): prettify_axes(ax)

# Accrual
if len(cum_by_month):
    ax1.plot(cum_by_month.index, cum_by_month.values, color=palette[0], linewidth=2)
    ax1.fill_between(cum_by_month.index, cum_by_month.values, step="pre", color=palette[0], alpha=0.12)
    ax1.set_title("Cohort accrual over time", pad=12, color=fg)
    ax1.set_xlabel("Calendar month", color=fg)
    ax1.set_ylabel("Cumulative patients", color=fg)
    ax1.tick_params(axis='x', labelrotation=30)
    ax1.yaxis.set_major_locator(MaxNLocator(integer=True))

# Age histogram
if len(age_vals):
    ax2.hist(age_vals, bins=30, edgecolor="white", linewidth=0.8, color=palette[1], alpha=0.85)
    ax2.set_title("Age at AF index", pad=12, color=fg)
    ax2.set_xlabel("Age (years)", color=fg)
    ax2.set_ylabel("Count", color=fg)
    ax2.yaxis.set_major_locator(MaxNLocator(integer=True))

# Sex bars
if len(sex_counts):
    bars = ax3.bar(sex_counts.index.astype(str), sex_counts.values, edgecolor="white", linewidth=0.8,
                   color=[palette[3] if i%2==0 else palette[4] for i in range(len(sex_counts))], alpha=0.9)
    ax3.set_title("Sex distribution", pad=12, color=fg)
    ax3.set_ylabel("Count", color=fg)
    ax3.tick_params(axis='x', rotation=0)
    ax3.yaxis.set_major_locator(MaxNLocator(integer=True))
    # annotate bars
    for b in bars:
        ax3.text(b.get_x() + b.get_width()/2, b.get_height(),
                 f"{int(b.get_height()):,}", ha='center', va='bottom', fontsize=8, color=fg)

# C) Bottom row: Pacemaker ECDF, BMI hist, Antiplatelet bar
ax4 = fig.add_subplot(gs[2, 0])  # Time to PM (ECDF)
ax5 = fig.add_subplot(gs[2, 1])  # BMI histogram
ax6 = fig.add_subplot(gs[2, 2])  # Antiplatelet bar
for ax in (ax4, ax5, ax6): prettify_axes(ax)

# PM ECDF
if len(pm_days_nonnull):
    x = np.sort(pm_days_nonnull.values); y = np.arange(1, len(x)+1) / len(x)
    ax4.step(x, y, where='post', color=palette[2], linewidth=2)
    ax4.set_title("Time to pacemaker (ECDF)", pad=12, color=fg)
    ax4.set_xlabel("Days from AF index", color=fg)
    ax4.set_ylabel("Cumulative proportion", color=fg)
    ax4.set_ylim(0, 1.02)

# BMI hist + category shares as text
if len(bmi_vals):
    ax5.hist(bmi_vals, bins=30, edgecolor="white", linewidth=0.8, color=palette[4], alpha=0.85)
    ax5.set_title("BMI near AF index", pad=12, color=fg)
    ax5.set_xlabel("BMI (kg/m²)", color=fg)
    ax5.set_ylabel("Count", color=fg)
    ax5.yaxis.set_major_locator(MaxNLocator(integer=True))
    if len(bmi_share):
        txt = "\n".join([f"{k}: {v:.1%}" for k,v in bmi_share.items()])
        ax5.text(0.98, 0.98, txt, ha='right', va='top', transform=ax5.transAxes, fontsize=9, color=fg)

# Antiplatelet therapy within 6 mo (bar)
ap_vals = cohort['ON_ANTIPLATELET_WITHIN_6MO'].fillna(0).astype(int) 
ax6.set_title("Antiplatelet ≤6 mo after AF", pad=12, color=fg)
ax6.set_ylabel("Patients", color=fg)
ax6.yaxis.set_major_locator(MaxNLocator(integer=True))
if not ap_vals.empty:
    ap_counts = ap_vals.value_counts().reindex([0, 1], fill_value=0)
    bars = ax6.bar(["No", "Yes"], ap_counts.values,
                   color=[palette[7], palette[0]],
                   edgecolor="white", linewidth=0.8, alpha=0.95)
    for b, lab in zip(bars, ["No", "Yes"]):
        ax6.text(b.get_x()+b.get_width()/2, b.get_height(),
                 f"{int(b.get_height()):,}", ha='center', va='bottom', fontsize=8, color=fg)
else:
    ax6.text(0.5, 0.5, "No data", ha='center', va='center', color=fg)
    ax6.set_xticks([]); ax6.set_yticks([])

# Title + footnote
fig.suptitle("Atrial Fibrillation Cohort on AIR-MS",
             fontsize=40, fontweight='bold', y=0.98, color=fg)
fig.text(0.01, 0.01,
         fill("Notes: AF index is first qualifying condition date. Antiplatelet exposure window "
              "is AF_INDEX_DATE to +6 months. BMI is the nearest measurement within the ±6-month window "
              "around AF index (if available).", 120),
         ha='left', va='bottom', fontsize=8, color="#374151")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()